In [149]:
from pathlib import Path
import pandas as pd
from pypdf import PdfReader
import time
import re
import unicodedata
from transformers import AutoTokenizer
import numpy as np

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc import DocItemLabel,TableItem, TextItem
from docling_core.types.doc import ContentLayer

# 1.import des données et setup

In [3]:
RACINE = Path.cwd().parent          # notebooks/ -> racine du repo
DOSSIER_DATA = RACINE / "data"      # gitignored
PDF_REGLES = DOSSIER_DATA / "core-rules" / "fr-warhammer40k_regles_de_base_01_06_2026.pdf"   # à adapter
DOSSIER_SORTIE = DOSSIER_DATA / "interim"
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)

assert PDF_REGLES.exists(), f"PDF introuvable : {PDF_REGLES}"

vérifier que le texte est bien sélectionnable

In [4]:
lecteur = PdfReader(PDF_REGLES)
nb_car = [len(p.extract_text() or "") for p in lecteur.pages]
stats = pd.Series(nb_car, index=range(1, len(nb_car) + 1), name="nb_caracteres")

print(f"{len(stats)} pages")
print(stats.describe())
print("Pages quasi vides (< 50 car.) :", stats[stats < 50].index.tolist())

88 pages
count      88.000000
mean     1853.977273
std      1098.675169
min         0.000000
25%       974.250000
50%      2127.000000
75%      2757.000000
max      4008.000000
Name: nb_caracteres, dtype: float64
Pages quasi vides (< 50 car.) : [3, 6, 7, 26, 27, 44, 45, 59, 60, 61, 75, 76, 77]


On vérifie s'il est possible de lire les signets. ça serait très utile  pour le découpage en chapitres.

In [5]:
def parcourir_signets(signets, lecteur, niveau=0):
    """Aplatit l'arbre des signets du PDF.

    Args:
        signets: liste (imbriquée) renvoyée par PdfReader.outline.
        lecteur: le PdfReader, pour résoudre les numéros de page.
        niveau: profondeur courante dans l'arbre.

    Yields:
        dict avec titre, niveau et page (numérotée à partir de 1).
    """
    for entree in signets:
        if isinstance(entree, list):   # une liste = les enfants du signet précédent
            yield from parcourir_signets(entree, lecteur, niveau + 1)
        else:
            yield {
                "titre": entree.title,
                "niveau": niveau,
                "page": lecteur.get_destination_page_number(entree) + 1,
            }

df_signets = pd.DataFrame(parcourir_signets(lecteur.outline, lecteur))
print(f"{len(df_signets)} signets, niveaux : {sorted(df_signets['niveau'].unique()) if len(df_signets) else '—'}")
df_signets.head(40)

0 signets, niveaux : —


""


# 2. conversion via docling

conversion Docling, mise en cache

In [12]:
options = PdfPipelineOptions(do_ocr=False, do_table_structure=True)
convertisseur = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=options)}
)

debut = time.perf_counter()
resultat = convertisseur.convert(PDF_REGLES)  # , page_range=(1, 20) pour un test rapide
doc = resultat.document
print(f"Conversion : {time.perf_counter() - debut:.0f} s — statut {resultat.status}")

NOM_JSON_REGLES = "fr-warhammer40k_regles_de_base_docling.json"

CACHE_JSON = DOSSIER_SORTIE / NOM_JSON_REGLES
doc.save_as_json(CACHE_JSON)
(DOSSIER_SORTIE / "regles_docling.md").write_text(doc.export_to_markdown(), encoding="utf-8")

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 16859.46it/s]


Conversion : 4 s — statut ConversionStatus.SUCCESS


152326

Pour les sessions suivantes, on recharge le cache au lieu de reconvertir :

In [7]:
doc = DoclingDocument.load_from_json(CACHE_JSON)

titres détectés par Docling, avec leur page.

In [13]:
for item, _ in doc.iterate_items():
    print(item.text)
    print("============")
    print(item.label.value)
    print("======================")
    print(getattr(item, "level", None))
    break

Dans les ténèbres d'un lointain futur, l'Humanité est au bord de l'extinction. L'Imperium à l'envergure galactique, assailli de toute part par des xénos voraces, des traîtres infâmes et des démons du Warp, compte une fois encore sur ses plus grands héros pour repousser l'obscurité qui menace de l'engloutir.
text
None


In [14]:
titres = [
    {
        "texte": item.text,
        "label": item.label.value,
        "niveau_docling": getattr(item, "level", None),
        "page": item.prov[0].page_no if item.prov else None,
    }
    for item, _ in doc.iterate_items()
    if item.label in (DocItemLabel.TITLE, DocItemLabel.SECTION_HEADER)
]
df_titres = pd.DataFrame(titres)
print(df_titres["niveau_docling"].value_counts(dropna=False))
df_titres.head(40)

niveau_docling
1    298
Name: count, dtype: int64


,texte,label,niveau_docling,page
0,INTRODUCTION,section_header,1,4
1,"WARHAMMER 40,000 : L'APPLI",section_header,1,5
2,►Exemple de Référence d'Appli,section_header,1,5
3,RÈGLES ÉLÉMENTAIRES,section_header,1,7
4,ARMÉES 01.01,section_header,1,8
5,UNITÉS ET FIGURINES 01.02,section_header,1,8
6,JOUEUR ACTIF ET  JOUEUR ADVERSE 01.03,section_header,1,8
7,MESURER LES DISTANCES 01.04,section_header,1,9
8,DÉS 01.05,section_header,1,9
9,JETS DE COMMANDEMENT 01.06,section_header,1,9


# 3. chunking maison

## 3.1 découpage en chapitres, sections etc..

In [52]:
STRUCTURE_LIVRE = [
    {"titre":"INTRODUCTION", "page_debut" : 4,"page_fin":5,"sections": [
        {"num":"00","titre":"introduction", "page_debut": 4}

    ]},
    {"titre": "RÈGLES ÉLÉMENTAIRES", "page_debut": 6, "page_fin": 25, "sections": [
        {"num": "01", "titre": "Concepts de base", "page_debut": 8},
        {"num": "02", "titre": "Fiches techniques", "page_debut": 10},
        {"num": "03", "titre": "Mouvement", "page_debut": 12},
        {"num": "04", "titre": "Effectuer des attaques", "page_debut": 16},
        {"num": "05", "titre": "Séquence d'attaque", "page_debut": 18},
        {"num": "06", "titre": "Autres concepts", "page_debut": 24},
    ]},
    {"titre": "LE ROUND DE BATAILLE", "page_debut": 26, "page_fin": 43, "sections": [
        {"num": "07", "titre": "Le round de bataille", "page_debut": 28},
        {"num": "08", "titre": "Phase de commandement", "page_debut": 30},
        {"num": "09", "titre": "Phase de mouvement", "page_debut": 32},
        {"num": "10", "titre": "Phase de tir", "page_debut": 34},
        {"num": "11", "titre": "Phase de charge", "page_debut": 36},
        {"num": "12", "titre": "Phase de combat", "page_debut": 38},
    ]},
    {"titre": "CHAMPS DE BATAILLE ET TACTIQUES", "page_debut": 44, "page_fin": 59, "sections": [
        {"num": "13", "titre": "Terrain", "page_debut": 46},
        {"num": "14", "titre": "Objectif", "page_debut": 52},
        {"num": "15", "titre": "Stratagèmes", "page_debut": 54},
        {"num": "16", "titre": "Actions", "page_debut": 58},
    ]},
    {"titre": "RÈGLES AVANCÉES", "page_debut": 60, "page_fin": 75, "sections": [
        {"num": "17", "titre": "Monstres et véhicules", "page_debut": 62},
        {"num": "18", "titre": "Transport", "page_debut": 64},
        {"num": "19", "titre": "Unités attachées", "page_debut": 66},
        {"num": "20", "titre": "Réserves stratégiques", "page_debut": 68},
        {"num": "21", "titre": "Vol et élan", "page_debut": 70},
        {"num": "22", "titre": "Autres règles et aptitudes", "page_debut": 72},
        {"num": "23", "titre": "Aérodynes", "page_debut": 74},
    ]},
    {"titre": "RÉFÉRENCES", "page_debut": 76, "page_fin": 89, "sections": [
        {"num": "24", "titre": "Aptitudes de base", "page_debut": 78},
        {"num": None, "titre": "Appendices des règles", "page_debut": 86},
        {"num": None, "titre": "Index des règles de base", "page_debut": 89},
    ]},
]

fonctions de localisation et de classement des titres

In [115]:
MOTIF_CODE = re.compile(
    r"^(?:\d+\.\s+)?(?P<titre>.+?)\s+(?P<sec>\d{2})\.(?P<sous>\d{2})"
    r"(?:\s+(?P<cout>\d+\s*PC))?$"   # coût de stratagème facultatif, ex. « 1PC »
)


def inspecter_page(page: int, motif: str | None = None) -> None:
    """Affiche tout ce que Docling a produit sur une page, images et toutes couches comprises.

    Args:
        page: numéro de page à inspecter.
        motif: texte à rechercher dans la couche texte brute du PDF (ex. "10.04").
    """
    if motif:
        brut = lecteur.pages[page - 1].extract_text()
        pos = brut.find(motif)
        print(f"pypdf — '{motif}' {'trouvé' if pos >= 0 else 'ABSENT'} :",
              repr(brut[max(0, pos - 60): pos + 60]) if pos >= 0 else "")
    print(f"\n{'type':<18}{'label':<16}{'couche':<11}{'parent':<16}texte")
    for item, _ in doc.iterate_items(traverse_pictures=True,
                                     included_content_layers=set(ContentLayer),
                                     page_no=page):
        if isinstance(item, TableItem):
            texte = item.export_to_markdown(doc=doc)
        else:
            texte = getattr(item, "text", "")
        parent = item.parent.cref if item.parent else "-"
        print(f"{type(item).__name__:<18}{item.label.value:<16}"
              f"{item.content_layer.value:<11}{parent:<16}{texte[:70]!r}")

def preparer_titre(texte: str) -> str:
    """Rend un titre comparable : NFKC, sans caractères de format invisibles, espaces simples.

    Args:
        texte: texte brut du titre.

    Returns:
        Titre nettoyé des artefacts d'encodage (utilisé pour la détection et comme métadonnée).
    """
    t = unicodedata.normalize("NFKC", texte)
    t = "".join(c for c in t if c.isspace() or unicodedata.category(c) not in ("Cc", "Cf"))
    return re.sub(r"\s+", " ", t).strip()

def normaliser(texte: str) -> str:
    """Forme de comparaison : sans accents, minuscules, espaces simples."""
    t = unicodedata.normalize("NFKD", preparer_titre(texte))
    t = "".join(c for c in t if not unicodedata.combining(c))
    return t.casefold()

def analyser_titre(texte: str, section_num: str | None = None) -> dict:
    """Classe un titre détecté par Docling.

    Args:
        texte: texte brut du titre.
        section_num: section déduite de la page, pour écarter les renvois.

    Returns:
        dict avec type (sous_section | renvoi_suspect | pseudo_titre | titre_chapitre
        | titre_section | sous_partie), titre et code.
    """
    t = preparer_titre(texte)
    if m := MOTIF_CODE.match(t):
        code = f"{m['sec']}.{m['sous']}"
        if section_num is not None and m["sec"] != section_num:
            return {"type": "renvoi_suspect", "titre": t, "code": code}
        return {"type": "sous_section", "titre": m["titre"], "code": code}
    if t.endswith(":"):
        return {"type": "pseudo_titre", "titre": t, "code": None}
    n = re.sub(r"^\d+\s*", "", normaliser(t))
    if n in TITRES_CHAPITRES:
        return {"type": "titre_chapitre", "titre": t, "code": None}
    if n in TITRES_SECTIONS:
        return {"type": "titre_section", "titre": t, "code": None}
    return {"type": "sous_partie", "titre": t, "code": None}


TITRES_CHAPITRES = {normaliser(c["titre"]) for c in STRUCTURE_LIVRE}
TITRES_SECTIONS = {normaliser(s["titre"]) for c in STRUCTURE_LIVRE for s in c["sections"]}

def localiser(page: int | None) -> dict:
    """Déduit chapitre et section à partir du numéro de page.

    Args:
        page: numéro de page Docling (1-indexé), ou None.

    Returns:
        dict avec chapitre, section_num et section_titre.
    """
    vide = {"chapitre": None, "section_num": None, "section_titre": None}
    if page is None:
        return vide
    chapitre = next((c for c in STRUCTURE_LIVRE if c["page_debut"] <= page <= c["page_fin"]), None)
    if chapitre is None:
        nom = "LIMINAIRE" if page < STRUCTURE_LIVRE[0]["page_debut"] else "HORS SOMMAIRE"
        return {**vide, "chapitre": nom}
    candidates = [s for s in chapitre["sections"] if s["page_debut"] <= page]
    if not candidates:  # pages d'ouverture du chapitre, avant sa première section
        return {**vide, "chapitre": chapitre["titre"], "section_titre": "(ouverture de chapitre)"}
    section = candidates[-1]
    return {"chapitre": chapitre["titre"], "section_num": section["num"], "section_titre": section["titre"]}

def est_dans_image(item, doc) -> bool:
    """Indique si un élément est rattaché (directement ou non) à une image Docling.

    Args:
        item: élément Docling.
        doc: le DoclingDocument, pour remonter la chaîne des parents.

    Returns:
        True si un ancêtre de l'élément est une image.
    """
    parent = item.parent
    while parent is not None:
        if parent.cref.startswith("#/pictures"):
            return True
        parent = parent.resolve(doc).parent
    return False


construction du tableau à plat

In [105]:
LABELS_TITRE = (DocItemLabel.TITLE, DocItemLabel.SECTION_HEADER)

lignes, reperes = [], []  # reperes = titres structurels, gardés pour les diagnostics
etat = {"code": None, "sous_section": None, "sous_partie": None}
section_courante = None

for ordre, (item, _) in enumerate(doc.iterate_items(traverse_pictures=True)):
    if not isinstance(item, (TextItem, TableItem)):
        continue  # images, groupes : ignorés pour ce test

    page = item.prov[0].page_no if item.prov else None
    loc = localiser(page)

    # Changement de section (déduit de la page) -> on repart d'un état vierge
    cle_section = (loc["chapitre"], loc["section_titre"])
    if cle_section != section_courante:
        section_courante = cle_section
        etat = dict.fromkeys(etat)

    type_contenu = "table" if isinstance(item, TableItem) else item.label.value

    if isinstance(item, TextItem) and item.label in LABELS_TITRE:
        a = analyser_titre(item.text, loc["section_num"])
        if a["type"] != "pseudo_titre":
            reperes.append({"ordre": ordre, "page": page, **a, "section_num_page": loc["section_num"]})
        if a["type"] == "sous_section":
            etat = {"code": a["code"], "sous_section": a["titre"], "sous_partie": None}
            continue
        if a["type"] in ("sous_partie", "renvoi_suspect"):  # un renvoi reste une simple sous-partie
            etat["sous_partie"] = a["titre"]
            continue
        if a["type"] in ("titre_chapitre", "titre_section"):
            continue
        type_contenu = "pseudo_titre"  # gardé comme contenu ordinaire

    texte = item.export_to_markdown(doc=doc) if isinstance(item, TableItem) else item.text
    lignes.append({"ordre": ordre, **loc, **etat, "page": page,
                   "type_contenu": type_contenu,
                   "dans_image": est_dans_image(item, doc),
                   "texte": texte})

df_items = pd.DataFrame(lignes)
df_reperes = pd.DataFrame(reperes)
print(f"{len(df_items)} éléments de contenu, {len(df_reperes)} titres structurels")
df_items.head(20)

1321 éléments de contenu, 319 titres structurels


,ordre,chapitre,section_num,section_titre,code,sous_section,sous_partie,page,type_contenu,dans_image,texte
0,0,LIMINAIRE,NaN,NaN,NaN,NaN,NaN,1,text,False,"Dans les ténèbres d'un lointain futur, l'Human..."
1,1,LIMINAIRE,NaN,NaN,NaN,NaN,NaN,1,text,False,Il n'y a pas de paix. Pas de répit. Pas de par...
2,2,LIMINAIRE,NaN,NaN,NaN,NaN,NaN,2,table,False,| Chaque section et sous-section possède ici u...
3,4,INTRODUCTION,00,introduction,NaN,NaN,NaN,4,text,False,Bienvenue dans les Règles de Base de Warhammer...
4,5,INTRODUCTION,00,introduction,NaN,NaN,NaN,4,text,False,"Warhammer 40,000 est un jeu de bataille sur ta..."
5,6,INTRODUCTION,00,introduction,NaN,NaN,NaN,4,text,False,Une bataille dure en général cinq rounds à l'i...
6,7,INTRODUCTION,00,introduction,NaN,NaN,NaN,5,text,False,"Chaque partie de Warhammer 40,000 s'inscrit da..."
7,8,INTRODUCTION,00,introduction,NaN,NaN,NaN,5,text,False,Les missions demanderont aux joueurs de rassem...
8,9,INTRODUCTION,00,introduction,NaN,NaN,NaN,5,text,False,Quelle que soit la faction que vous souhaitez ...
9,10,INTRODUCTION,00,introduction,NaN,NaN,NaN,5,text,False,"Enfin, avant de disputer une partie de Warhamm..."


In [106]:
df_reperes

,ordre,page,type,titre,code,section_num_page
0,3,4,titre_chapitre,INTRODUCTION,NaN,00
1,13,5,sous_partie,"WARHAMMER 40,000 : L'APPLI",NaN,00
2,17,5,sous_partie,►Exemple de Référence d'Appli,NaN,00
3,24,7,titre_chapitre,RÈGLES ÉLÉMENTAIRES,NaN,NaN
4,27,8,sous_section,ARMÉES,01.01,01
...,...,...,...,...,...,...
314,1758,87,sous_partie,VOIR AUSSI,NaN,NaN
315,1759,87,sous_partie,EFFECTIF INITIAL ET DEMI-EFFECTIF,NaN,NaN
316,1766,88,sous_partie,OBJECTIFS N'ÉTANT PAS DANS UNE ZONE DE TERRAIN,NaN,NaN
317,1769,88,sous_partie,RESSUSCITÉ,NaN,NaN


### diagnostics

In [107]:
# 1. Vue d'ensemble : taille et étendue de chaque section
resume = (
    df_items.groupby(["chapitre", "section_num", "section_titre"], dropna=False, sort=False)
    .agg(nb_elements=("texte", "size"),
         nb_caracteres=("texte", lambda s: s.str.len().sum()),
         page_min=("page", "min"), page_max=("page", "max"),
         nb_codes=("code", "nunique"))
)
display(resume)

nb_elements  \
chapitre                        section_num section_titre                             
LIMINAIRE                       NaN         NaN                                   3   
INTRODUCTION                    00          introduction                         13   
RÈGLES ÉLÉMENTAIRES             NaN         (ouverture de chapitre)               1   
                                01          Concepts de base                     42   
                                02          Fiches techniques                   110   
                                03          Mouvement                            96   
                                04          Effectuer des attaques               41   
                                05          Séquence d'attaque                  123   
                                06          Autres concepts                      17   
LE ROUND DE BATAILLE            07          Le round de bataille                 22   
                                08          Phase de commandement                28   
                                09          Phase de mouvement                   57   
                                10          Phase de tir                         52   
                                11          Phase de charge                      41   
                                12          Phase de combat                      92   
CHAMPS DE BATAILLE ET TACTIQUES 13          Terrain                              88   
                                14          Objectif                             17   
                                15          Stratagèmes                         125   
                                16          Actions                              31   
RÈGLES AVANCÉES                 NaN         (ouverture de chapitre)               1   
                                17          Monstres et véhicules                19   
                                18          Transport                            41   
                                19          Unités attachées                     18   
                                20          Réserves stratégiques                24   
                                21          Vol et élan                          36   
                                22          Autres règles et aptitudes           16   
                                23          Aérodynes                            11   
RÉFÉRENCES                      24          Aptitudes de base                   122   
                                NaN         Appendices des règles                34   

                                                                        nb_caracteres  \
chapitre                        section_num section_titre                               
LIMINAIRE                       NaN         NaN                                  8999   
INTRODUCTION                    00          introduction                         3569   
RÈGLES ÉLÉMENTAIRES             NaN         (ouverture de chapitre)                 1   
                                01          Concepts de base                     4818   
                                02          Fiches techniques                    5195   
                                03          Mouvement                            6642   
                                04          Effectuer des attaques               4333   
                                05          Séquence d'attaque                  11506   
                                06          Autres concepts                      3197   
LE ROUND DE BATAILLE            07          Le round de bataille                 2335   
                                08          Phase de commandement                2714   
                                09          Phase de mouvement                   4542   
                                10          Phase de tir                         4228   
                                11     

In [111]:
# 2. Cohérence code / page : le préfixe du code doit égaler la section déduite de la page
codes = df_reperes[df_reperes["type"] == "sous_section"]
incoherents = codes[codes["code"].str[:2] != codes["section_num_page"]]
print(f"\n⚠️ {len(incoherents)} code(s) incohérent(s) avec la page")
display(incoherents[["code", "titre", "page", "section_num_page"]])


⚠️ 0 code(s) incohérent(s) avec la page


,code,titre,page,section_num_page


In [112]:
# 3. Continuité et doublons des codes
for sec, g in codes.groupby(codes["code"].str[:2]):
    nums = sorted({int(c[3:]) for c in g["code"]})
    manquants = sorted(set(range(1, max(nums) + 1)) - set(nums))
    if manquants:
        print(f"⚠️ Section {sec} : sous-sections absentes {manquants}")
doublons = codes["code"].value_counts()
print("Codes en double :", doublons[doublons > 1].to_dict())

Codes en double : {'15.11': 2}


In [113]:
# 4. Décalage de pages : le 1er code de chaque section tombe-t-il sur sa page de début ?
attendu = {s["num"]: s["page_debut"] for c in STRUCTURE_LIVRE for s in c["sections"] if s["num"]}
premiere_page = codes.groupby(codes["code"].str[:2])["page"].min()
ecarts = {sec: (int(p), attendu.get(sec)) for sec, p in premiere_page.items() if p != attendu.get(sec)}
print("\nSections dont le 1er code n'est pas sur la page attendue (trouvée, attendue) :", ecarts)

# 5. Titres de chapitre / section détectés par Docling (contrôle croisé)
display(df_reperes[df_reperes["type"].isin(["titre_chapitre", "titre_section"])][["type", "titre", "page"]])


Sections dont le 1er code n'est pas sur la page attendue (trouvée, attendue) : {}


,type,titre,page
0,titre_chapitre,INTRODUCTION,4
3,titre_chapitre,RÈGLES ÉLÉMENTAIRES,7
70,titre_section,EFFECTUER DES ATTAQUES,20
98,titre_chapitre,LE ROUND DE BATAILLE,27
103,titre_section,PHASE DE COMMANDEMENT,29
104,titre_section,PHASE DE MOUVEMENT,29
105,titre_section,PHASE DE TIR,29
106,titre_section,PHASE DE CHARGE,29
107,titre_section,PHASE DE COMBAT,29
167,titre_chapitre,CHAMPS DE BATAILLE ET TACTIQUES,45


In [ ]:
mask = (df_items["chapitre"]=="CHAMPS DE BATAILLE ET TACTIQUES")  & (df_items["section_num"]=="15") & (df_items["code"]=="15.01")
df_items[mask]

In [103]:
inspecter_page(56, motif="15.02")

pypdf — '15.02' trouvé : 'RELANCE DE COMMANDEMENT 15.02 1PC\nSTRATAGÈME DE BASE\nUn grand commandant peut même p'

type              label           couche     parent          texte
SectionHeaderItem section_header  body       #/body          'RELANCE DE COMMANDEMENT 15.02 1PC'
SectionHeaderItem section_header  body       #/body          'STRATAGÈME DE BASE'
TextItem          text            body       #/body          'Un grand commandant peut même plier les caprices du destin à sa volont'
TextItem          text            body       #/body          "QUAND : À n'importe quelle phase, juste après que vous avez fait un de"
ListItem          list_item       body       #/groups/120    "Jet d'avance"
ListItem          list_item       body       #/groups/120    '▪Jet de charge'
ListItem          list_item       body       #/groups/120    '▪Jet de dégâts'
ListItem          list_item       body       #/groups/120    '▪Jet de risque'
ListItem          list_item       body       #/groups/120    

## 3.2 nettoyage simple

éléments très court

In [ ]:
courts = df_items.loc[df_items["texte"].str.len() < 25, "texte"]
print(courts.value_counts().head(40))

normalisation du texte

In [158]:
REMPLACEMENTS = str.maketrans({
    # Ligatures typographiques
    "ﬀ": "ff", "ﬁ": "fi", "ﬂ": "fl", "ﬃ": "ffi", "ﬄ": "ffl",
    # Apostrophes
    "’": "'", "‘": "'", "ʼ": "'",
    # Guillemets anglais et double prime (pouces) -> guillemet droit
    "“": '"', "”": '"', "„": '"', "″": '"',
})
MOTIF_PUCE = re.compile(r"^[▪▫•●]\s*")

def normaliser_texte(texte: str, est_tableau: bool = False) -> str:
    """Normalise le texte d'un élément sans NFKC, pour préserver ½, pouces, etc.

    Args:
        texte: texte brut de l'élément.
        est_tableau: True pour un tableau Markdown (on préserve les retours à la ligne).

    Returns:
        Texte normalisé.
    """
    t = "".join(c for c in texte if c.isspace() or unicodedata.category(c) not in ("Cc", "Cf"))
    t = t.translate(REMPLACEMENTS)
    if est_tableau:
        t = re.sub(r"[^\S\n]+", " ", t)  # blancs sauf retours à la ligne
        return "\n".join(ligne.strip() for ligne in t.split("\n")).strip()
    t = re.sub(r"\s+", " ", t).strip()
    return MOTIF_PUCE.sub("- ", t)

annotation

In [159]:
MOTIF_CODE_CITE = re.compile(r"\b\d{2}\.\d{2}\b")
ZONES_HORS_REGLES = {"Index des règles de base", "(ouverture de chapitre)", "introduction"}
DEBUT_REGLE_STRATAGEME = ("QUAND", "CIBLE", "EFFET", "RESTRICTIONS")
PONCTUATION_FINALE = (".", ":", ";", "!", "?")

def ressemble_a_une_etiquette(t: str) -> bool:
    """Étiquette de schéma : tout en majuscules / sans lettres, ou ≤ 3 mots sans ponctuation finale.

    Args:
        t: texte normalisé.

    Returns:
        True si le texte a la forme d'une étiquette plutôt que d'un fragment de phrase.
    """
    return t.upper() == t or (len(t.split()) <= 3 and not t.endswith(PONCTUATION_FINALE))


def motif_ligne(ligne: pd.Series) -> str | None:
    """Motif d'exclusion d'un élément (première règle qui s'applique), ou None.

    Args:
        ligne: une ligne de df_items, texte déjà normalisé.

    Returns:
        Nom du motif, ou None si l'élément est conservé.
    """
    t = ligne["texte"]
    if ligne["chapitre"] == "LIMINAIRE" or ligne["section_titre"] in ZONES_HORS_REGLES:
        return "zone_hors_regles"
    if not any(c.isalnum() for c in t):  # ½ compte comme alphanumérique : '½"' n'est pas vide
        return "sans_contenu"
    if re.fullmatch(r"\d{2}", t) and t == ligne["section_num"]:
        return "numero_section"
    if t.isdigit() and ligne["page"] == int(t):
        return "numero_page"
    if re.fullmatch(r"\d+\s*PC", t):
        return "cout_isole"

    zone_voir_aussi = str(ligne["sous_partie"]).startswith("VOIR AUSSI")
    renvoi_code = t.startswith("- ") and re.search(r"\d{2}\.\d{2}$", t)
    if renvoi_code or (zone_voir_aussi and len(t) <= 50):
        return "renvoi"
    if (ligne["dans_image"] and ligne["type_contenu"] != "table"
            and len(t) < 25 and ":" not in t and ressemble_a_une_etiquette(t)):
        return "etiquette_image"

    return None

def annoter_nettoyage(df: pd.DataFrame) -> pd.DataFrame:
    """Normalise le texte et annote chaque élément avec un éventuel motif d'exclusion.

    Rien n'est supprimé : les colonnes ajoutées permettent d'auditer puis de filtrer.

    Args:
        df: df_items issu de la cellule 8.

    Returns:
        Copie de df avec texte_brut, texte (normalisé), codes_cites, motif_exclusion, a_relire.
    """
    df = df.copy()
    if "texte_brut" not in df:  # la cellule peut être relancée sans écraser le texte d'origine
        df["texte_brut"] = df["texte"]
    df["texte"] = [normaliser_texte(t, est_tableau=(tc == "table"))
                   for t, tc in zip(df["texte_brut"], df["type_contenu"])]
    df["codes_cites"] = df["texte"].map(MOTIF_CODE_CITE.findall)

    # Règles élément par élément
    df["motif_exclusion"] = df.apply(motif_ligne, axis=1)

    # Lore des stratagèmes : texte avant la première ligne QUAND/CIBLE/… de chaque stratagème
    dans_strat = df["sous_partie"].eq("STRATAGÈME DE BASE") & df["motif_exclusion"].isna()
    debut_regle = df["texte"].str.startswith(DEBUT_REGLE_STRATAGEME) & dans_strat
    regle_commencee = debut_regle.groupby(df["code"].fillna("_")).cummax()
    lore = dans_strat & ~regle_commencee & df["type_contenu"].eq("text")
    df.loc[lore, "motif_exclusion"] = "lore_stratageme"

        # Doublons : même code et même texte, mais sur une autre page que la 1re occurrence.
    # Les répétitions sur une même page (logigrammes, Q/R, tableaux) sont légitimes.
    cle = ["code", "texte"]
    premiere_page = df.groupby(cle, dropna=False)["page"].transform("min")
    doublon = (df.duplicated(subset=cle, keep="first")
               & df["page"].ne(premiere_page)
               & df["motif_exclusion"].isna())
    df.loc[doublon, "motif_exclusion"] = "doublon"

    df["a_relire"] = df["motif_exclusion"].isna() & (df["texte"].str.len() < 15)
    return df

In [160]:
df_items = annoter_nettoyage(df_items)

In [161]:
df_items

,ordre,chapitre,section_num,section_titre,code,sous_section,sous_partie,page,type_contenu,dans_image,texte,texte_brut,codes_cites,motif_exclusion,a_relire
0,0,LIMINAIRE,NaN,NaN,NaN,NaN,NaN,1,text,False,"Dans les ténèbres d'un lointain futur, l'Human...","Dans les ténèbres d'un lointain futur, l'Human...",[],zone_hors_regles,False
1,1,LIMINAIRE,NaN,NaN,NaN,NaN,NaN,1,text,False,Il n'y a pas de paix. Pas de répit. Pas de par...,Il n'y a pas de paix. Pas de répit. Pas de par...,[],zone_hors_regles,False
2,2,LIMINAIRE,NaN,NaN,NaN,NaN,NaN,2,table,False,| Chaque section et sous-section possède ici u...,| Chaque section et sous-section possède ici u...,[],zone_hors_regles,False
3,4,INTRODUCTION,00,introduction,NaN,NaN,NaN,4,text,False,Bienvenue dans les Règles de Base de Warhammer...,Bienvenue dans les Règles de Base de Warhammer...,[],zone_hors_regles,False
4,5,INTRODUCTION,00,introduction,NaN,NaN,NaN,4,text,False,"Warhammer 40,000 est un jeu de bataille sur ta...","Warhammer 40,000 est un jeu de bataille sur ta...",[],zone_hors_regles,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1316,1781,RÉFÉRENCES,NaN,Appendices des règles,NaN,NaN,FAQ,88,text,False,R : Non.,R : Non.,[],NaN,True
1317,1782,RÉFÉRENCES,NaN,Appendices des règles,NaN,NaN,FAQ,88,text,False,Q : Une unité qui est éligible pour effectuer ...,Q : Une unité qui est éligible pour effectuer ...,[],NaN,False
1318,1783,RÉFÉRENCES,NaN,Appendices des règles,NaN,NaN,FAQ,88,text,False,"R : Non. Parfois, une unité peut devenir engag...","R : Non. Parfois, une unité peut devenir engag...",[],NaN,False
1319,1784,RÉFÉRENCES,NaN,Appendices des règles,NaN,NaN,FAQ,88,text,False,Q : Une unité peut-elle embarquer dans un TRAN...,Q : Une unité peut-elle embarquer dans un TRAN...,[],NaN,False


audit

In [162]:
# 1. Volume par motif
display(df_items["motif_exclusion"].value_counts(dropna=False).rename("nb_elements"))

# 2. Échantillon par motif : vérifier qu'aucune règle n'est emportée
for motif, g in df_items.dropna(subset=["motif_exclusion"]).groupby("motif_exclusion"):
    print(f"\n=== {motif} ({len(g)}) ===")
    for t in g["texte"].sample(min(8, len(g)), random_state=0):
        print("  ·", t[:90])

# 3. Signalés à relire (courts, sans motif)
print("\n=== à relire ===")
print(df_items.loc[df_items["a_relire"], "texte"].value_counts().head(30))

# 4. Contrôle des caractères sensibles : seul '"' doit augmenter (guillemets unifiés)
SENSIBLES = ["½", '"', "+", "×", "«", "»", "…", "-", "/"]
controle = pd.DataFrame(
    {c: [df_items["texte_brut"].str.count(re.escape(c)).sum(),
         df_items["texte"].str.count(re.escape(c)).sum()] for c in SENSIBLES},
    index=["brut", "normalisé"],
)
display(controle)

# 5. Mots coupés en fin de ligne restants (à inspecter, pas corrigés automatiquement)
coupes = df_items["texte"].str.contains(r"[a-zà-ÿ]- [a-zà-ÿ]")
print(f"\n{coupes.sum()} élément(s) avec une coupure suspecte")
print(df_items.loc[coupes, "texte"].head(5).tolist())

motif_exclusion
NaN                 997
etiquette_image     199
renvoi               47
numero_section       30
zone_hors_regles     18
lore_stratageme      10
doublon               9
cout_isole            9
sans_contenu          2
Name: nb_elements, dtype: int64


=== cout_isole (9) ===
  · 2PC
  · 1PC
  · 1PC
  · 1PC
  · 1PC
  · 1PC
  · 1PC
  · 1PC

=== doublon (9) ===
  · Quand vous choisissez des cibles de charge, vous pouvez choisir une ou plusieurs unités en
  · Ensuite, chaque joueur consulte sa mission ; si l'un ou les deux joueurs ont rempli un ou 
  · Résolvez d'abord les règles déclenchées à ce stade autres que les règles de mission.
  · Bondir pour Défendre : Quand vous choisissez des cibles de charge, vous pouvez seulement c
  · Scannez le code ci-dessous pour installer dès maintenant l'appli Warhammer 40,000.
  · Quand vous faites le jet de charge, si le résultat est supérieur à 6 (après modificateurs)
  · EFFET : Résolvez une charge avec votre unité (11.02). Ce faisant, avant de faire le jet de
  · L'unité ROUGE attaque. On choisit les armes suivantes avec lesquelles effectuer des attaqu

=== etiquette_image (199) ===
  · APTITUDES
  · A
  · 2.
  · 2"
  · 15"
  · PLEIN
  · ►Tirs au Dé
  · DÉBUT DE LA PHASE DE TIR

=== lore_stratag

,½,"""",+,×,«,»,…,-,/
brut,5,129,59,0,0,0,0,1492,64
normalisé,5,129,59,0,0,0,0,1600,64



4 élément(s) avec une coupure suspecte
["Cette unité a un effectif initial de 5. Il lui reste deux figurines, elle est donc en dessous de son demi- effectif et on doit faire un jet d'ébranlement pour elle.", "Ce VÉHICULE a un effectif initial de 1 et une caracté- ristique de PV de 11. Il lui reste 3 points de vie, il est donc en dessous de son demi-effectif et on doit faire un jet d'ébranlement pour lui.", "Cette figurine d'INFANTERIE ne se déplace pas par- dessus ni à travers des éléments de terrain. Son mouvement à travers la zone de terrain indiquée se fait donc sans encombre.", "Il y a une zone de terrain occultante entre toutes les figuri- nes des unités A et D, donc l'une n'est pas visible de l'autre."]


In [163]:
for motif in ["renvoi", "etiquette_image"]:
    g = df_items[df_items["motif_exclusion"] == motif]
    print(f"\n=== {motif} : les 10 plus longs ===")
    for t in g.loc[g["texte"].str.len().nlargest(10).index, "texte"]:
        print("  ·", t[:100])


=== renvoi : les 10 plus longs ===
  · - Se Déplacer à la Verticale 13.06
  · - Mouvement de Consolidation 12.08
  · - Mouvement de Débarquement 18.04
  · - Aptitudes d'Équipement 22.04
  · APTITUDES DES UNITÉS ATTACHÉES
  · - Phase de Commandement 08.00
  · - Réserves Stratégiques 20.00
  · - Mouvement de Retraite 09.07
  · - Mouvement d'Insertion 12.03
  · - Mouvement d'Éclaireur 24.32

=== etiquette_image : les 10 plus longs ===
  · DÉBUT DE LA PHASE DE TIR
  · FIN DE LA PHASE DE TIR
  · ►Séquençage de Règles
  · ►Réussite Automatique
  · OPTIONS D'ÉQUIPEMENT
  · COMPOSITION D'UNITÉ
  · Objectif de Terrain
  · Objectif de Terrain
  · Objectif de Terrain
  · Objectif de Terrain


## 3.3 Choix pour le découpage en chunks

Cible / max	 --> Effet attendu

250 / 400	--> Plus précis, mais davantage de règles coupées en morceaux

400 / 600 (spec) -->	Compromis standard ; la plupart des sous-sections devraient tenir entières

600 / 800	--> Presque rien n'est coupé, au prix de vecteurs plus « dilués » pour les grosses sous-sections


In [164]:
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")  # tokenizer seul, pas le modèle

Reconstruction du texte des unités

In [152]:

def nb_tokens(texte: str) -> int:
    """Nombre de tokens BGE-M3 d'un texte (sans tokens spéciaux)."""
    return len(tokenizer(texte, add_special_tokens=False)["input_ids"])

def valeur(x):
    """Convertit les NaN pandas en None."""
    return None if pd.isna(x) else x

def construire_prefixe(chapitre, section_num, section_titre, code, sous_section) -> str:
    """Construit le chemin de contexte « Chapitre > NN Section > NN.NN Sous-section ».

    Returns:
        Préfixe sans les niveaux absents.
    """
    niveaux = [
        chapitre,
        " ".join(p for p in (section_num, section_titre) if p),
        " ".join(p for p in (code, sous_section) if p),
    ]
    return " > ".join(n for n in niveaux if n)

def rendre_unite(g: pd.DataFrame, prefixe: str) -> str:
    """Reconstruit le texte Markdown d'un groupe d'éléments, titres de sous-parties réinsérés.

    Args:
        g: éléments du groupe, triés dans l'ordre de lecture.
        prefixe: chemin de contexte placé en tête.

    Returns:
        Texte tel qu'il sera embarqué.
    """
    lignes, sous_partie_courante = [prefixe, ""], None
    for sp, tc, texte in zip(g["sous_partie"], g["type_contenu"], g["texte"]):
        sp = valeur(sp)
        if sp and sp != sous_partie_courante:
            lignes += ["", f"### {sp}"]
            sous_partie_courante = sp
        lignes += ["", texte, ""] if tc == "table" else [texte]
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lignes)).strip()

mesure par sous-section et par sous-partie

In [145]:
df_contenu = df_items[df_items["motif_exclusion"].isna()].sort_values("ordre")
df_contenu_motif = df_items[df_items["motif_exclusion"].notna()].sort_values("ordre")

In [150]:
np.unique(df_contenu_motif["motif_exclusion"])

array(['cout_isole', 'doublon', 'etiquette_image', 'lore_stratageme',
       'numero_page', 'numero_section', 'renvoi', 'sans_contenu',
       'zone_hors_regles'], dtype=object)

In [153]:
CLES_UNITE = ["chapitre", "section_num", "section_titre", "code", "sous_section"]

unites = []
for cles, g in df_contenu.groupby(CLES_UNITE, dropna=False, sort=False):
    cles = [valeur(c) for c in cles]
    prefixe = construire_prefixe(*cles)
    tailles_sp = [nb_tokens(rendre_unite(sg, prefixe))
                  for _, sg in g.groupby(g["sous_partie"].fillna("∅"), sort=False)]
    unites.append({
        **dict(zip(CLES_UNITE, cles)),
        "nb_elements": len(g),
        "page_min": g["page"].min(), "page_max": g["page"].max(),
        "nb_tokens": nb_tokens(rendre_unite(g, prefixe)),
        "nb_sous_parties": len(tailles_sp),
        "max_tokens_sous_partie": max(tailles_sp),
        "contient_table": (g["type_contenu"] == "table").any(),
    })
df_unites = pd.DataFrame(unites)
print(f"{len(df_unites)} unités (sous-sections + introductions de section)")

181 unités (sous-sections + introductions de section)


In [154]:
df_unites

,chapitre,section_num,section_titre,code,sous_section,nb_elements,page_min,page_max,nb_tokens,nb_sous_parties,max_tokens_sous_partie,contient_table
0,INTRODUCTION,00,introduction,NaN,NaN,12,4,5,929,3,737,False
1,RÈGLES ÉLÉMENTAIRES,01,Concepts de base,NaN,NaN,1,8,8,77,1,77,False
2,RÈGLES ÉLÉMENTAIRES,01,Concepts de base,01.01,ARMÉES,1,8,8,91,1,91,False
3,RÈGLES ÉLÉMENTAIRES,01,Concepts de base,01.02,UNITÉS ET FIGURINES,5,8,8,223,1,223,False
4,RÈGLES ÉLÉMENTAIRES,01,Concepts de base,01.03,JOUEUR ACTIF ET JOUEUR ADVERSE,5,8,8,267,1,267,False
...,...,...,...,...,...,...,...,...,...,...,...,...
176,RÉFÉRENCES,24,Aptitudes de base,24.26,[TIR UNIQUE],4,85,85,181,1,181,False
177,RÉFÉRENCES,24,Aptitudes de base,24.37,[TORRENT],2,85,85,77,1,77,False
178,RÉFÉRENCES,24,Aptitudes de base,24.23,[TOUCHES FATALES],3,85,85,190,1,190,False
179,RÉFÉRENCES,24,Aptitudes de base,24.36,[TOUCHES SOUTENUES],3,85,85,187,1,187,False


rapport d'audit

In [155]:
# 1. Distribution des tailles
print(df_unites["nb_tokens"].describe(percentiles=[.5, .75, .9, .95]).round(0))

# 2. Combien d'unités dépassent chaque seuil candidat
for seuil in (250, 400, 600, 800):
    trop = df_unites["nb_tokens"] > seuil
    print(f"> {seuil:>3} tokens : {trop.sum():>3} unités ({trop.mean():.0%})")

# 3. Pour chaque budget max : le découpage par sous-parties suffit-il ?
for seuil in (400, 600, 800):
    gros = df_unites[df_unites["nb_tokens"] > seuil]
    print(f"Budget {seuil} : {len(gros)} unités à découper, dont "
          f"{(gros['max_tokens_sous_partie'] > seuil).sum()} avec une sous-partie seule encore trop grosse")

# 4. Les 15 plus grosses unités
display(df_unites.nlargest(15, "nb_tokens")[
    ["code", "sous_section", "section_titre", "nb_tokens", "nb_sous_parties",
     "max_tokens_sous_partie", "contient_table", "page_min", "page_max"]])

# 5. Les très petites unités, et la taille des tableaux
print("\nUnités < 50 tokens :", (df_unites["nb_tokens"] < 50).sum())
tables = df_contenu.loc[df_contenu["type_contenu"] == "table", "texte"].map(nb_tokens)
print("Tableaux (tokens) :", tables.describe().round(0).to_dict())

count     181.0
mean      242.0
std       286.0
min        36.0
50%       157.0
75%       292.0
90%       492.0
95%       643.0
max      2454.0
Name: nb_tokens, dtype: float64
> 250 tokens :  56 unités (31%)
> 400 tokens :  28 unités (15%)
> 600 tokens :  12 unités (7%)
> 800 tokens :   4 unités (2%)
Budget 400 : 28 unités à découper, dont 10 avec une sous-partie seule encore trop grosse
Budget 600 : 12 unités à découper, dont 1 avec une sous-partie seule encore trop grosse
Budget 800 : 4 unités à découper, dont 0 avec une sous-partie seule encore trop grosse


,code,sous_section,section_titre,nb_tokens,nb_sous_parties,max_tokens_sous_partie,contient_table,page_min,page_max
180,NaN,NaN,Appendices des règles,2454,12,411,True,86,88
30,05.04,INFLIGER DES DÉGÂTS,Séquence d'attaque,2164,12,451,True,19,23
18,03.01,DÉPLACER DES UNITÉS,Mouvement,1085,10,226,False,12,14
0,NaN,NaN,introduction,929,3,737,False,4,5
82,13.06,TERRAIN ET MOUVEMENT,Terrain,800,4,304,False,48,49
25,04.03,RÉSOUDRE LES ATTAQUES,Effectuer des attaques,786,6,494,False,17,17
69,12.03,MOUVEMENT D'INSERTION,Phase de combat,746,6,301,False,38,40
87,13.11,PLEIN,Terrain,661,3,339,False,50,51
81,13.05,DENSE,Terrain,659,4,269,False,46,48
120,19.04,APTITUDES DES UNITÉS ATTACHÉES,Unités attachées,643,3,365,True,67,67



Unités < 50 tokens : 11
Tableaux (tokens) : {'count': 5.0, 'mean': 194.0, 'std': 56.0, 'min': 110.0, '25%': 181.0, '50%': 190.0, '75%': 242.0, 'max': 247.0}


Chunk	Contenu	Tokens	Pourquoi
1	A + B	300	ajouter C dépasserait 400, on ferme
2	C	451	trop grosse pour être regroupée, mais sous 600
3	D + E	340	on regroupe ce qui reste


Chaque chunk commence par le même préfixe (… > 05.04 INFLIGER DES DÉGÂTS), suivi des titres de ses sous-parties.

## 3.4 Découpage en chunk

paramètres

In [165]:
import uuid

CIBLE_TOKENS = 400   # taille visée quand on regroupe des sous-parties
MAX_TOKENS = 600     # au-delà, une sous-section est découpée
EDITION = "11e"
SOURCE = "livre_regles_principal"
ESPACE_UUID = uuid.uuid5(uuid.NAMESPACE_URL, "assistant-regles-w40k")

def taille(elements: pd.DataFrame, prefixe: str) -> int:
    """Taille en tokens du texte rendu d'un groupe d'éléments, préfixe compris."""
    return nb_tokens(rendre_unite(elements, prefixe))

les trois fonctions de découpage

In [166]:
def blocs_sous_parties(g: pd.DataFrame) -> list[pd.DataFrame]:
    """Découpe une unité en blocs de sous-parties contiguës, dans l'ordre de lecture.

    Args:
        g: éléments d'une unité, triés par ordre.

    Returns:
        Liste de blocs ; les éléments sans sous-partie forment leur propre bloc.
    """
    cle = g["sous_partie"].fillna("∅")
    numero_bloc = (cle != cle.shift()).cumsum()
    return [bloc for _, bloc in g.groupby(numero_bloc, sort=False)]


def decouper_paragraphes(bloc: pd.DataFrame, prefixe: str) -> list[pd.DataFrame]:
    """Filet de sécurité : découpe un bloc trop gros aux frontières d'éléments, avec chevauchement.

    Le dernier élément d'un morceau est répété en tête du suivant (paragraphe entier,
    jamais une coupe en milieu de phrase). Un élément seul hors budget reste entier.

    Args:
        bloc: éléments d'une sous-partie dépassant MAX_TOKENS.
        prefixe: chemin de contexte.

    Returns:
        Liste de morceaux.
    """
    morceaux, courant = [], []
    for i in range(len(bloc)):
        element = bloc.iloc[[i]]
        if taille(element, prefixe) > MAX_TOKENS:  # ex. très gros tableau : gardé entier
            if courant:
                morceaux.append(pd.concat(courant))
            morceaux.append(element)
            courant = []
            continue
        if courant and taille(pd.concat(courant + [element]), prefixe) > CIBLE_TOKENS:
            morceaux.append(pd.concat(courant))
            # Chevauchement seulement si le morceau fermé contient plusieurs éléments
            # (sinon il serait entièrement inclus dans le suivant).
            chevauchement = courant[-1]
            avec_chevauchement = len(courant) > 1 and \
                taille(pd.concat([chevauchement, element]), prefixe) <= MAX_TOKENS
            courant = [chevauchement, element] if avec_chevauchement else [element]
        else:
            courant.append(element)
    if courant:
        morceaux.append(pd.concat(courant))
    return morceaux


def decouper_unite(g: pd.DataFrame, prefixe: str) -> list[pd.DataFrame]:
    """Découpe une sous-section en morceaux respectant le budget.

    Niveau 1 : l'unité tient dans MAX_TOKENS -> un seul morceau.
    Niveau 2 : sinon, regroupement de sous-parties contiguës jusqu'à CIBLE_TOKENS.
    Niveau 3 : une sous-partie seule > MAX_TOKENS -> découpage par paragraphes.

    Args:
        g: éléments de l'unité, triés par ordre.
        prefixe: chemin de contexte.

    Returns:
        Liste de morceaux (DataFrames d'éléments).
    """
    if taille(g, prefixe) <= MAX_TOKENS:
        return [g]

    morceaux, courant = [], []
    for bloc in blocs_sous_parties(g):
        if taille(bloc, prefixe) > MAX_TOKENS:
            if courant:
                morceaux.append(pd.concat(courant))
                courant = []
            morceaux.extend(decouper_paragraphes(bloc, prefixe))
            continue
        if courant and taille(pd.concat(courant + [bloc]), prefixe) > CIBLE_TOKENS:
            morceaux.append(pd.concat(courant))
            courant = [bloc]
        else:
            courant.append(bloc)
    if courant:
        morceaux.append(pd.concat(courant))
    return morceaux

construction des chunks et de leurs métadonnées, puis sauvegarde en JSONL

In [167]:
def construire_chunk(morceau: pd.DataFrame, cles: tuple, prefixe: str,
                     partie: int, nb_parties: int) -> dict:
    """Assemble le texte et les métadonnées d'un chunk.

    Args:
        morceau: éléments du chunk.
        cles: (chapitre, section_num, section_titre, code, sous_section).
        prefixe: chemin de contexte.
        partie: rang du chunk dans son unité (à partir de 1).
        nb_parties: nombre de chunks de l'unité.

    Returns:
        dict prêt à être sérialisé en JSONL.
    """
    chapitre, section_num, section_titre, code, sous_section = cles
    texte = rendre_unite(morceau, prefixe)
    n = nb_tokens(texte)
    ordres = morceau["ordre"].tolist()
    codes_cites = sorted({c for liste in morceau["codes_cites"] for c in liste} - {code})
    return {
        "id": str(uuid.uuid5(ESPACE_UUID, f"{SOURCE}|{ordres[0]}|{ordres[-1]}")),
        "texte": texte,
        "chapitre": chapitre,
        "section_num": section_num,
        "section_titre": section_titre,
        "code": code,
        "sous_section": sous_section,
        "sous_parties": list(dict.fromkeys(v for v in morceau["sous_partie"] if pd.notna(v))),
        "page_debut": int(morceau["page"].min()),
        "page_fin": int(morceau["page"].max()),
        "type_contenu": "table" if (morceau["type_contenu"] == "table").all() else "regle",
        "edition": EDITION,
        "source": SOURCE,
        "codes_cites": codes_cites,
        "partie": partie,
        "nb_parties": nb_parties,
        "nb_tokens": n,
        "hors_budget": n > MAX_TOKENS,
        "ordres": ordres,
    }


In [170]:
CLES_UNITE

['chapitre', 'section_num', 'section_titre', 'code', 'sous_section']

In [168]:
df_contenu = df_items[df_items["motif_exclusion"].isna()].sort_values("ordre")

chunks = []
for cles, g in df_contenu.groupby(CLES_UNITE, dropna=False, sort=False):
    cles = tuple(valeur(c) for c in cles)
    prefixe = construire_prefixe(*cles)
    morceaux = decouper_unite(g, prefixe)
    chunks += [construire_chunk(m, cles, prefixe, i, len(morceaux))
               for i, m in enumerate(morceaux, start=1)]

df_chunks = pd.DataFrame(chunks)

FICHIER_CHUNKS = DOSSIER_SORTIE / "chunks.jsonl"
df_chunks.to_json(FICHIER_CHUNKS, orient="records", lines=True, force_ascii=False)
print(f"✓ {len(df_chunks)} chunks écrits dans {FICHIER_CHUNKS.relative_to(RACINE)}")

✓ 207 chunks écrits dans data/interim/chunks.jsonl


validation

In [169]:
# 1. Tailles
print(df_chunks["nb_tokens"].describe(percentiles=[.5, .9, .95]).round(0))
print("Chunks hors budget :", df_chunks["hors_budget"].sum())

# 2. Unités découpées et nombre de parties
decoupees = df_chunks[df_chunks["nb_parties"] > 1]
display(decoupees.groupby(["section_titre", "code"], dropna=False, sort=False)
        .agg(nb_parties=("partie", "size"), tokens=("nb_tokens", list)))

# 3. Couverture : chaque élément conservé doit apparaître dans au moins un chunk
tous_ordres = pd.Series([o for liste in df_chunks["ordres"] for o in liste])
manquants = set(df_contenu["ordre"]) - set(tous_ordres)
print(f"\nÉléments non couverts : {len(manquants)}")
print(f"Éléments répétés (chevauchement) : {(tous_ordres.value_counts() > 1).sum()}")

# 4. Relecture : unités découpées emblématiques + échantillon aléatoire
def afficher_chunk(c) -> None:
    """Affiche un chunk avec ses principales métadonnées."""
    print("=" * 80)
    print(f"[{c.code or '—'}] partie {c.partie}/{c.nb_parties} · p. {c.page_debut}-{c.page_fin} · "
          f"{c.nb_tokens} tokens · {c.type_contenu} · renvois {c.codes_cites}")
    print("-" * 80)
    print(c.texte)

echantillon = pd.concat([
    df_chunks[df_chunks["code"] == "05.04"].head(3),
    df_chunks[df_chunks["code"] == "15.11"],
    df_chunks.sample(10, random_state=0),
])
for c in echantillon.itertuples():
    afficher_chunk(c)

count    207.0
mean     210.0
std      132.0
min       36.0
50%      181.0
90%      391.0
95%      454.0
max      568.0
Name: nb_tokens, dtype: float64
Chunks hors budget : 0


nb_parties  \
section_titre          code                
Mouvement              03.01           4   
Effectuer des attaques 04.03           2   
Séquence d'attaque     05.04           7   
Phase de combat        12.03           3   
Terrain                13.05           2   
                       13.06           3   
                       13.11           2   
Stratagèmes            15.11           2   
Monstres et véhicules  17.03           2   
Unités attachées       19.04           3   
Appendices des règles  NaN             8   

                                                                tokens  
section_titre          code                                             
Mouvement              03.01                      [335, 383, 320, 122]  
Effectuer des attaques 04.03                                [494, 321]  
Séquence d'attaque     05.04       [451, 288, 360, 326, 287, 358, 274]  
Phase de combat        12.03                           [361, 320, 111]  
Terrain                13.05                                [345, 335]  
                       13.06                           [304, 273, 275]  
                       13.11                                [343, 339]  
Stratagèmes            15.11                                [549, 108]  
Monstres et véhicules  17.03                                [257, 387]  
Unités attachées       19.04                           [153, 365, 187]  
Appendices des règles  NaN    [251, 391, 392, 176, 261, 379, 277, 411]


Éléments non couverts : 0
Éléments répétés (chevauchement) : 0
[05.04] partie 1/7 · p. 19-19 · 451 tokens · regle · renvois []
--------------------------------------------------------------------------------
RÈGLES ÉLÉMENTAIRES > 05 Séquence d'attaque > 05.04 INFLIGER DES DÉGÂTS

Le joueur adverse résout la séquence suivante pour chaque jet de sauvegarde, du résultat le plus bas au plus élevé, jusqu'à avoir résolu toutes les attaques ou avoir détruit toutes les figurines de l'unité ; dans ce dernier cas, les attaques en excès sont perdues.
Choisissez la Figurine : Choisissez 1 figurine dans le groupe d'allocation actuel (voir à droite) ; il doit s'agir d'une figurine qui a perdu un ou plusieurs points de vie si possible.
Vérifiez le Jet de Sauvegarde : Pour chaque résultat, vérifiez si l'attaque inflige des dégâts ou échoue selon la première condition ci-dessous qui s'applique :
Résolvez les Dégâts : Si l'attaque inflige des dégâts, la figurine choisie perd autant de points de vie que

In [ ]:
df_chunks